### To do list :
- [ ] Add missing metabolites to biomass equation

### Setting up


In [ ]:
%pip install cobra

In [1]:
import cobra
import numpy as np
import pandas as pd
from cobra.io import load_model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite, Gene

Upload model

In [76]:
model = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\Biomass Equation\\biomass_update_2.sbml')

In [3]:
model1 = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Solarfood_original.sbml')

In [86]:
model1 = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\PhD\\Data\\Models\\Reference_GSMs\\iSynCJ816.xml')

No objective coefficients in model. Unclear what should be optimized


In [6]:
copy_of_model = model.copy()

In [69]:
# set reactions bounds for a specific reaction 
reaction = copy_of_model.reactions.get_by_id('Growth')
reaction.bounds = -0.0, 0.0

In [10]:
c_uptake =10

In [90]:
copy_of_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0002956,0,0.00%
cl_e,EX_cl_e,0.0002956,0,0.00%
co2_e,EX_co2_e,2.5,1,100.00%
cobalt2_e,EX_cobalt2_e,5.679E-06,0,0.00%
cu2_e,EX_cu2_e,4.026E-05,0,0.00%
fe2_e,EX_fe2_e,0.0008627,0,0.00%
h2_e,EX_h2_e,15.06,0,0.00%
k_e,EX_k_e,0.01108,0,0.00%
mg2_e,EX_mg2_e,0.0004926,0,0.00%
mn2_e,EX_mn2_e,3.924E-05,0,0.00%


In [5]:

# Set ammonia uptake rate to 1 mmol/(gDCW·h)
ammonia_uptake_rxn = copy_of_model.reactions.get_by_id('EX_nh4_e')  # Replace with actual ammonia exchange reaction ID


# Maximize the ATPase reaction
atp_hydrolysis_rxn = copy_of_model.reactions.get_by_id('ATPM')  # Replace with actual ATP hydrolysis reaction ID
copy_of_model.objective = atp_hydrolysis_rxn

# Perform optimization
solution = copy_of_model.optimize()

# Check the flux through the ATP hydrolysis and ammonia uptake reactions
atp_flux = solution.fluxes[atp_hydrolysis_rxn.id]
ammonia_flux = solution.fluxes[ammonia_uptake_rxn.id]

print(f'ATP Hydrolysis Flux: {atp_flux} mmol ATP/(gDCW·h)')
print(f'Ammonia Uptake Flux: {ammonia_flux} mmol NH3/(gDCW·h)')

ATP Hydrolysis Flux: 14.000000000000002 mmol ATP/(gDCW·h)
Ammonia Uptake Flux: 0.0 mmol NH3/(gDCW·h)


# Manual Curation

## Adding metabolites

In [212]:
new_metabolite = Metabolite(
    id='ethyln_c',     # Metabolite ID
    name='Ethylnitronate',     # Metabolite Name
    formula='C2H4NO2',         # Chemical Formula
    compartment='c'            # Compartment (e.g., cytoplasm)
)

new_metabolite.charge = -1   

copy_of_model.add_metabolites([new_metabolite])

print(new_metabolite.formula)

C2H4NO2


## Reactions & genes handling

### Search metabolite

#### Choose a metabolite and see the fluxes of each reactions it is involved in it

In [88]:
model1.metabolites.get_by_id('hdec_c').summary()


Percent,Flux,Reaction,Definition
Percent,Flux,Reaction,Definition


In [79]:
search_word = 'hepdcoa'
# Iterate over metabolites in the model
for metabolite in copy_of_model.metabolites:
    if search_word in metabolite.id:
        # Print metabolite information
        print(metabolite.id)
        print(metabolite.charge)
        print(metabolite.formula)
        print(metabolite.name)
        print(metabolite.compartment)
        print(metabolite.annotation)
        
        # Get related reactions
        related_reactions = list(metabolite.reactions)
        
        # Get genes related to the reactions
        related_genes = set()
        for reaction in related_reactions:
            related_genes.update(reaction.genes)
        
        # Print related reactions and associated genes
        print(f"Related Reactions: {', '.join(r.id for r in related_reactions)}")
        print(f"Genes Associated: {', '.join(g.id for g in related_genes)}\n")

### Search reaction

In [63]:
search_word = 'PROHEXADECA'

for reaction in copy_of_model.reactions:
    if search_word in reaction.id:
        print(f"Reaction ID: {reaction.id}")
        print(f"Name: {reaction.name}")
        print(f"Equation: {reaction.reaction}")
        print(f"Lower Bound: {reaction.lower_bound}")
        print(f"Upper Bound: {reaction.upper_bound}")
        print(f"Genes: {[gene.id for gene in reaction.genes]}")

Reaction ID: PROHEXADECA
Name: Protonation of Hexadecanoate to Hexadecanoic acid
Equation: h_c + hdca_c --> hexadecacid_c
Lower Bound: 0.0
Upper Bound: 1000.0
Genes: []


### Check reaction

In [4]:
reaction_id_to_check = 'Growth'
reaction = model1.reactions.get_by_id(reaction_id_to_check)
print(f"Reaction ID: {reaction.id}")
print(f"Name: {reaction.name}")
print(f"Equation: {reaction.reaction}")
print(f"Lower Bound: {reaction.lower_bound}")
print(f"Upper Bound: {reaction.upper_bound}")
sum = 0
#reaction.name = '(S)-2-Acetolactate pyruvate-lyase (carboxylating)'
print("\nMetabolites and Stoichiometry:")
for metabolite, coefficient in reaction.metabolites.items():
    print(f"{metabolite.id:<15}: {coefficient:>25}  Name: {metabolite.name:<60}  Charge: {metabolite.charge:>3}  Formula: {metabolite.formula}")
    sum += coefficient * metabolite.charge
print(f'\nSum charge: {sum}')
print("\nAssociated Genes:")
for gene in reaction.genes:
    print(gene.id)


Reaction ID: Growth
Name: Biomass equation measured by SolarFoods
Equation: 0.000223 10fthf_c + 0.000223 2dmmql8_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.816872 ala__L_c + 0.000223 amet_c + 0.255798 arg__L_c + 0.252959 asn__L_c + 0.252959 asp__L_c + 29.190682 atp_c + 0.004952 ca2_c + 0.000223 chor_c + 0.004952 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000168 coa_c + 2.4e-05 cobalt2_c + 0.010737 ctp_c + 0.006744 cu2_c + 0.037472 cys__L_c + 0.014833 datp_c + 0.015499 dctp_c + 0.024005 dgtp_c + 0.013367 dttp_c + 0.000223 enter_c + 0.000223 fad_c + 0.006388 fe2_c + 0.007428 fe3_c + 0.644556 fru_c + 0.005772 gal_c + 0.117367 glc__D_c + 0.275321 gln__L_c + 0.275321 glu__L_c + 0.637966 gly_c + 0.000223 gthrd_c + 0.01659 gtp_c + 29.180423 h2o_c + 0.000223 hemeO_c + 0.093522 his__L_c + 0.245064 ile__L_c + 0.18569 k_c + 0.480939 leu__L_c + 0.257602 lys__L_c + 3.1e-05 malcoa_c + 0.00481 man_c + 0.142675 met__L_c + 0.008253 mg2_c + 0.000223 mlthf_c + 0.000658 mn2_c +

<b>We found out that the model contains one reaction of three for the polyhydroxybutyrate which is 'Acetoacetyl CoA reductase' </b>

### Modify bounds of reaction

In [96]:
copy_of_model.reactions.bounds = -1000.,1000.

### Check gene

In [ ]:

# Example gene ID to check
gene_id = 'AAFOLC_01125'

# Get the gene
gene = copy_of_model.genes.get_by_id(gene_id)

# Print information about the gene
print(f"Gene ID: {gene.id}")
print(f"Name: {gene.name}")

# Print reactions associated with the gene
print("\nReactions Associated with the Gene:")
for reaction in gene.reactions:
    print(f"Reaction ID: {reaction.id}")
    print(f"Name: {reaction.name}")
    print(f"Equation: {reaction.reaction}")
    print(f"Lower Bound: {reaction.lower_bound}")
    print(f"Upper Bound: {reaction.upper_bound}")

    # Print metabolites and stoichiometry
    print("\nMetabolites and Stoichiometry:")
    for metabolite, coefficient in reaction.metabolites.items():
        print(f"{metabolite.id}: {coefficient}")

Add a new gene to the model

In [38]:
if gene_id not in copy_of_model.genes:
    new_gene = Gene(id=gene_id)
    copy_of_model.genes.append(new_gene)
else:
    new_gene = copy_of_model.genes.get_by_id(gene_id)

Add gene to specific reaction

In [ ]:
reaction = copy_of_model.reactions.get_by_id('ACACDC')

reaction.gene_reaction_rule = gene_id
reaction.genes

### Add annotation to reaction

In [6]:
reaction = copy_of_model.reactions.get_by_id('ACACDC')
reaction.annotation["kegg.reaction"] = "R01366"
reaction.annotation["ec-code"] = "4.1.1.4"
reaction.annotation["metanetx.reaction"] = "MNXR95442"

print(reaction.annotation)

{'kegg.reaction': 'R01366', 'ec-code': '4.1.1.4', 'metanetx.reaction': 'MNXR95442'}


### Add annotation to gene

In [41]:
gene = copy_of_model.genes.get_by_id("AAFOLC_01125")
# Add annotations
gene.annotation["uniprot"] = "Q89CN4"

# Verify the annotation
print(gene.annotation)

{'uniprot': 'Q89CN4'}


### Add reaction

In [67]:

# more realistic fluxes by forcing TCA activity
# model.reactions.get_by_id('MDH').bounds = 0,0
# model.reactions.get_by_id('AKGMALtm').bounds = 0,0

ID = 'PROHEXADECE'
if ID not in copy_of_model.reactions:
    reaction = Reaction(ID)
    reaction.name = 'Protonation of Hexadecenoate to Hexadecenoic acid'
    reaction.lower_bound = 0.
    reaction.upper_bound = 1000.
    # reaction.annotation = {'biocyc':'4.2.3.27-RXN' , 'brenda': '4.2.3.27'}
    # HAA.annotation = {'biocyc':'CPD-9436', 'chebi':'35194', 'chemspider':'6309', 'inchi':' InChI=1S/C5H8/c1-4-5(2)3/h4H,1-2H2,3H3', 'kegg.compound':'C16521', 'pubchem':'6557'}    
    hdca = copy_of_model.metabolites.get_by_id('hdcea_c')
    hexedecacid = copy_of_model.metabolites.get_by_id('hexedecacid_c')
    co2 = copy_of_model.metabolites.get_by_id('co2_c')
    h = copy_of_model.metabolites.get_by_id('h_c')    
    reaction.add_metabolites({hdca:-1.,h:-1.,hexedecacid:1.})
    
    copy_of_model.add_reactions([reaction])

    print(copy_of_model.reactions.get_by_id(ID).id)
    print(copy_of_model.reactions.get_by_id(ID).reaction)



PROHEXADECE
h_c + hdcea_c --> hexedecacid_c


#### Add a sink reaction

In [32]:
copy_of_model.add_boundary(copy_of_model.metabolites.get_by_id('rb15bp_c'), type='sink',lb=0.0, ub=1000)

Reaction identifier,SK_rb15bp_c
Name,"D-Ribulose 1,5-bisphosphate sink"
Memory address,0x194c836c0d0
Stoichiometry,"rb15bp_c --> D-Ribulose 1,5-bisphosphate -->"
GPR,
Lower bound,0.0
Upper bound,1000


#### Run FBA with an objective function

In [62]:
from cobra.flux_analysis import pfba
# Set the sink reaction as the objective
copy_of_model.objective = "Growth"

# Perform FBA
solution = copy_of_model.optimize()
print(f"Yield: {solution.objective_value/co2_uptake}")

# Perform parsimonious FBA (optional, to minimize overall flux)
fba_solution = pfba(copy_of_model)
print(fba_solution.fluxes)

# Print the flux through the sink reaction for 3PG
print(f"Flux through the objective reaction: {fba_solution.fluxes['Growth']}")

Yield: 0.024372205498150633
EX_25dkglcn_e      0.000000
EX_2ameph_e        0.000000
EX_2m35mdntha_e    0.000000
EX_2pglyc_e        0.000000
EX_34dhbz_e        0.000000
                     ...   
5DH4DGLCD          0.000000
ETOHt              0.000000
RBPC               9.073133
SBP                3.072408
PRUK               9.073133
Name: fluxes, Length: 2364, dtype: float64
Flux through the objective reaction: 0.24372205498149893


#### Show the fluxes with a detailed description of the reaction

In [69]:
significant_reactions = [
    [
        copy_of_model.reactions[RctIdx].name, 
        solution.to_frame().iloc[RctIdx].name, 
        copy_of_model.reactions[RctIdx].reaction, 
        round(solution.fluxes[RctIdx], 2)
    ]
    for RctIdx in np.where(np.abs(solution.fluxes) > 3)[0]
]

# Create a DataFrame for better visualization
df = pd.DataFrame(significant_reactions, columns=["Reaction Name", "Reaction ID", "Reaction Equation", "Flux"])

# Print the DataFrame
df

C:\Users\adam-1y9pxi6q95ggpmx\AppData\Local\Temp\ipykernel_14784\2195648100.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  round(solution.fluxes[RctIdx], 2)


,Reaction Name,Reaction ID,Reaction Equation,Flux
0,EX_co2_e,EX_co2_e,co2_e <=>,-10.00
1,EX_h2_e,EX_h2_e,h2_e <=>,-67.07
2,EX_h2o_e,EX_h2o_e,h2o_e <=>,64.62
3,EX_o2_e,EX_o2_e,o2_e <=>,-23.57
4,Acetate-CoA ligase (ADP-forming),ACCOAL,atp_c + coa_c + ppa_c --> adp_c + pi_c + ppcoa_c,4.58
5,,APAT_1,atp_c + h_c + ppa_c <=> ppad_c + ppi_c,-3.96
6,ATP synthase (four protons for one ATP) (perip...,ATPS4rpp,adp_c + 4.0 h_p + pi_c <=> atp_c + h2o_c + 3.0...,47.19
7,CO2 transport via diffusion (extracellular to ...,CO2tex,co2_e <=> co2_p,10.00
8,CO2 transporter via diffusion (periplasm),CO2tpp,co2_p <=> co2_c,10.00
9,Cytochrome oxidase bd (ubiquinol-8: 2 protons)...,CYTBDpp,2.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 2.0 h_...,47.14


## Remove reaction

In [39]:
# Reaction ID to be removed
reaction_id = "SK_rb15bp_c"

# Remove the reaction from the model
copy_of_model.remove_reactions([reaction_id])

# Verify the removal
if reaction_id not in copy_of_model.reactions:
    print(f"Reaction {reaction_id} has been successfully removed.")
else:
    print(f"Failed to remove reaction {reaction_id}.")

Reaction SK_rb15bp_c has been successfully removed.


## Mass and charge balance

### Read excel files

In [70]:
file_path = 'C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\Modified reactions.xlsx'
file_path_1 = 'C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\Modified formula.xlsx'
file_path_2 = 'C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\Manually modified reactions.xlsx'
# Read Excel files into DataFrames
excel_data_charge = pd.read_excel(file_path)
excel_data_formula = pd.read_excel(file_path_1)
excel_data_Manualcuration = pd.read_excel(file_path_2)

### Check number of unbalanced reactions

In [71]:
used_model = copy_of_model

In [72]:
print(len(cobra.manipulation.validate.check_mass_balance(used_model)))
list_reactions = cobra.manipulation.validate.check_mass_balance(used_model)
for reaction, balance_dict in list_reactions.items():
    reaction_id = reaction.id
    charge_balance = balance_dict
    print(f"Reaction ID: {reaction_id}, Charge Balance: {charge_balance}")

6
Reaction ID: DHNAOT, Charge Balance: {'charge': 2.0}
Reaction ID: HDECH, Charge Balance: {'charge': 2.0}
Reaction ID: MECDPDH4E, Charge Balance: {'charge': -1.0}
Reaction ID: P9DH, Charge Balance: {'charge': 4.0}
Reaction ID: THZSN_1, Charge Balance: {'H': -1.0}
Reaction ID: ACACDC, Charge Balance: {'charge': -1.0, 'C': -1.0, 'H': -3.0, 'O': 1.0}


### Reaction information

In [13]:
reaction_id_to_check = 'Growth'
reaction = used_model.reactions.get_by_id(reaction_id_to_check)
print(f"Reaction ID: {reaction.id}")
print(f"Name: {reaction.name}")
print(f"Equation: {reaction.reaction}")
print(f"Lower Bound: {reaction.lower_bound}")
print(f"Upper Bound: {reaction.upper_bound}")
sum = 0
#reaction.name = '(S)-2-Acetolactate pyruvate-lyase (carboxylating)'
print("\nMetabolites and Stoichiometry:")
for metabolite, coefficient in reaction.metabolites.items():
    print(f"{metabolite.id:<15}: {coefficient:>25}  Name: {metabolite.name:<60}  Charge: {metabolite.charge:>3}  Formula: {metabolite.formula}")
    sum += coefficient * metabolite.charge
print(f'\nSum charge: {sum}')
print("\nAssociated Genes:")
for gene in reaction.genes:
    print(gene.id)

Reaction ID: Growth
Name: Biomass reaction
Equation: 0.000223 10fthf_c + 0.513689 ala__L_c + 0.000223 amet_c + 0.295792 arg__L_c + 0.241055 asn__L_c + 0.241055 asp__L_c + 54.124831 atp_c + 0.005205 ca2_c + 0.005205 cl_c + 0.000576 coa_c + 0.0001 cobalt2_c + 0.133508 ctp_c + 0.000709 cu2_c + 0.09158 cys__L_c + 0.026166 datp_c + 0.027017 dctp_c + 0.027017 dgtp_c + 0.026166 dttp_c + 0.000223 fad_c + 0.006715 fe2_c + 0.007808 fe3_c + 0.26316 gln__L_c + 0.26316 glu__L_c + 0.612638 gly_c + 0.215096 gtp_c + 48.601527 h2o_c + 0.094738 his__L_c + 0.290529 ile__L_c + 0.195193 k_c + 0.019456 kdo2lipid4_p + 0.450531 leu__L_c + 0.343161 lys__L_c + 0.153686 met__L_c + 0.008675 mg2_c + 0.000223 mlthf_c + 0.000691 mn2_c + 0.0001 mql8_c + 0.013894 murein5px4p_p + 0.001831 nad_c + 0.000447 nadp_c + 0.017868 pe160_c + 0.045946 pe160_p + 0.054154 pe161_c + 0.02106 pe161_p + 0.185265 phe__L_c + 0.000223 pheme_c + 0.221055 pro__L_c + 0.000223 pydx5p_c + 0.000223 ribflv_c + 0.215792 ser__L_c + 0.000223 sheme

### Add metabolite

In [66]:
new_metabolite = Metabolite(
    id='hexedecacid_c',     # Metabolite ID
    name='Hexadecenoic acid',     # Metabolite Name
    formula='C16H30O2',         # Chemical Formula
    compartment='c'            # Compartment (e.g., cytoplasm)
)

new_metabolite.charge = 0

copy_of_model.add_metabolites([new_metabolite])

print(new_metabolite.formula)

C16H30O2


In [221]:
reaction = copy_of_model.reactions.get_by_id('Growth')
metabolite_to_add = 'murein3px4p_p'
reaction.add_metabolites({metabolite_to_add: -0.000605})
copy_of_model.slim_optimize()

1.0554787605796536

Add modification to Excel file

In [65]:
modification =f'{metabolite_to_add} was added'
# Create a new row with the modification information
new_row = {'reaction ': reaction_id_to_check, 'modification': modification}
# Concatenate the new row to the DataFrame
excel_data_Manualcuration = pd.concat([excel_data_Manualcuration, pd.DataFrame([new_row])], ignore_index=True)

NameError: name 'metabolite_to_add' is not defined

### Remove metabolite

In [266]:
reaction = used_model.reactions.get_by_id(reaction_id_to_check)
metabolite_to_add = 'h_c'
reaction.subtract_metabolites({metabolite_to_add: 4.0})
print(reaction.reaction)

mqn8_c + pppg9_c --> mql8_c + ppp9_c


Add modification to Excel file

In [217]:
modification =f'{metabolite_to_add} was removed'
# Create a new row with the modification information
new_row = {'reaction ': reaction_id_to_check, 'modification': modification}
# Concatenate the new row to the DataFrame
excel_data_Manualcuration = pd.concat([excel_data_Manualcuration, pd.DataFrame([new_row])], ignore_index=True)

### Change charge of metabolite

In [18]:
metabolite_id = 'gam_c'
if metabolite_id in model.metabolites:
    metabolite = used_model.metabolites.get_by_id(metabolite_id)
    initial_charge = metabolite.charge  # Store the initial charge for reference
    metabolite.charge = 1
    metabolite.compartment = 'c'
    
    print(list(metabolite.reactions))  
    # Print information about the modified metabolite
    print(f"Modified metabolite: {metabolite.id}")
    print(f"Updated charge: {metabolite.charge}")
    print(f"Updated charge: {metabolite.compartment}")
else:
    print("Metabolite not found in the model.")

Metabolite not found in the model.


#### Add to excel file

In [155]:
# Create a dictionary containing information about the modified reaction
new_row = {'Reaction': reaction_id_to_check, 'metabolite changed': metabolite_id, 'initial charge': initial_charge, 'New charge': metabolite.charge}
excel_data_charge = pd.concat([excel_data_charge, pd.DataFrame([new_row])], ignore_index=True)
excel_data_charge

,Reaction,metabolite changed,initial charge,New charge
0,2AGPG161tipp,2agpg161_c,0,-1
1,3HACPH,r3hbACP_c,0,-1
2,3HAD140,tmrs2eACP_c,0,-1
3,4HOXPACt2pp_1,4hoxpac_p,0,-1
4,4OXPTNtpp,4oxptn_p,0,-1
5,4OXPTNtex,4oxptn_e,0,-1
6,4hoxpactex,4hoxpac_e,0,-1
7,6ATHAtexi,6atha_e,0,-1
8,AADa,aad_c,0,-1
9,ACLS_b,hethmpp_c,-1,-2


### Change formula of metabolite

In [256]:
metabolite_id = 'octadecacid_c'
metabolite = used_model.metabolites.get_by_id(metabolite_id)
initial_formula = metabolite.formula
metabolite.formula = 'C18H43O10'

#### Add to excel file

In [186]:
new_row_1 = {
    'Reaction': reaction_id_to_check,        # ID of the reaction associated with the metabolite change
    'metabolite changed': metabolite_id,     # ID of the metabolite that was changed
    'initial formula': initial_formula,      # Initial formula of the metabolite
    'New formula': metabolite.formula        # New formula of the metabolite after modification
}
excel_data_formula = pd.concat([excel_data_formula, pd.DataFrame([new_row_1])], ignore_index=False)

excel_data_formula = excel_data_formula.reset_index(drop=True)


In [ ]:
excel_data_formula

### Save updated files

#### Save Excel

In [264]:
# Save the updated DataFrame to the Excel file
excel_data_charge.to_excel(file_path, index=False)
excel_data_formula.to_excel(file_path_1, index=False)
excel_data_Manualcuration.to_excel(file_path_2, index=False)

#### Save model in SBML 

In [75]:
sbml_filename = "C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\biomass_update_2.sbml"
cobra.io.write_sbml_model(copy_of_model, sbml_filename)

## Exchange reaction

In [47]:
copy_of_model = model.copy()

In [89]:
# set exchange reactions bounds to zero if they significally affect the growth
ex_list = []

not_to_change_list = ['EX_ca2_e','EX_cl_e','EX_cobalt2_e','EX_cu2_e','EX_h2o_e','EX_h2_e','EX_fe_e','EX_fe2_e','EX_fe3_e'
                      ,'EX_k_e','EX_mg2_e','EX_mn2_e','EX_zn2_e','EX_o2_e','EX_so4_e','EX_pi_e','EX_nh4_e','EX_mobd_e','EX_ni2_e']

for element in copy_of_model.exchanges:
    reaction = copy_of_model.reactions.get_by_id(element.id)
    
    # Iterate over metabolites associated with the reaction
    for metabolite, coefficient in reaction.metabolites.items():
        if element.id not in not_to_change_list:
            # Set reaction bounds to zero
            reaction.bounds = 0.0, 0.0
            
            # Optimize the model
            solution = copy_of_model.slim_optimize()
            
            # Check if the solution is not zero
            if solution != 0.0:
                ex_list.append(reaction.id)
                
            # Print reaction information
            print(reaction.name)
            print(solution)

EX_25dkglcn_e
0.0
EX_2ameph_e
0.0
EX_2m35mdntha_e
0.0
EX_2pglyc_e
0.0
EX_34dhbz_e
0.0
EX_35dnta_e
0.0
EX_3cmp_e
0.0
EX_3h4atb_e
0.0
EX_4abut_e
0.0
EX_4abz_e
0.0
EX_4hba_e
0.0
EX_4hoxpac_e
0.0
EX_4hphac_e
0.0
EX_4hpro_DC_e
0.0
EX_4hpro_LT_e
0.0
EX_4oxptn_e
0.0
EX_5drib_e
0.0
EX_5mcsn_e
0.0
EX_6atha_e
0.0
EX_LalaDgluMdapDala_e
0.0
EX_LalaLglu_e
0.0
EX_Lcyst_e
0.0
EX_Ncbmpts_e
0.0
EX_R_3hhpa_e
0.0
EX_R_3hnonaa_e
0.0
EX_R_3hpba_e
0.0
EX_R_3hpdeca_e
0.0
EX_R_3hphpa_e
0.0
EX_R_3hpnona_e
0.0
EX_R_3hpocta_e
0.0
EX_R_3hpt_e
0.0
EX_R_3htd58e_e
0.0
EX_abt_e
0.0
EX_ac_e
0.0
EX_acald_e
0.0
EX_acgam_e
0.0
EX_acmana_e
0.0
EX_acnam_e
0.0
EX_acon_C_e
0.0
EX_actn__R_e
0.0
EX_ade_e
0.0
EX_agm_e
0.0
EX_ala__D_e
0.0
EX_ala__L_e
0.0
EX_alahis_e
0.0
EX_alaleu_e
0.0
EX_alathr_e
0.0
EX_alatrp_e
0.0
EX_anhgm_e
0.0
EX_apc_e
0.0
EX_arab__L_e
0.0
EX_arbt6p_e
0.0
EX_arbt_e
0.0
EX_arg__L_e
0.0
EX_ascb__L_e
0.0
EX_asn__L_e
0.0
EX_aso3_e
0.0
EX_aso4_e
0.0
EX_asp__L_e
0.0
EX_bhb_e
0.0
EX_bz_e
0.0
EX_carn_e
0.0
EX_cd2_e

In [90]:
# change bounds of the exchange reactions
for element in reac:
    reaction = copy_of_model.reactions.get_by_id(element)
    for metabolite, coefficient in reaction.metabolites.items():
        print(reaction.name)
        print(metabolite.name)
        print(metabolite.formula)
        reaction.bounds = -1000.0 , 1000.0

In [ ]:
# set reactions bounds for a specific reaction 
reaction = copy_of_model.reactions.get_by_id('EX_co2_e')
reaction.bounds = -1000.0, 1000.0

In [82]:
# set reactions bounds for a specific reaction 
reaction = copy_of_model.reactions.get_by_id('EX_h2_e')
reaction.bounds = -10.0, 10.0

In [88]:
solution = copy_of_model.slim_optimize()

copy_of_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
Metabolite,Reaction,Flux,C-Number,C-Flux


## Biomass Equation Curation

Updating the coefficient and adding metabolites from the Solar Food Biomass Equation

In [27]:
file_path = 'C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\Biomass_equation.xlsx'
# Read Excel files into DataFrames
pd_biomass = pd.read_excel(file_path)
pd_biomass

,Biomass_equation,Unnamed: 1,molar mass g/mol,measured mmol/g,Unmeasured mmol/g,unmeasured g/g,measured g/g,GAM,normalized g/g,normalized_mmol/g
0,10fthf_c,10-Formyltetrahydrofolate,471.43007,NaN,-0.000223,-0.000105,NaN,NaN,0.000000,-0.000223
1,pydx5p_c,Pyridoxal 5'-phosphate,245.12862,NaN,-0.000223,-0.000055,NaN,NaN,0.000000,-0.000223
2,murein3px4p_p,"Two disacharide linked murein units, tripeptid...",1750.69809,NaN,-0.000605,-0.001059,NaN,NaN,0.000000,-0.000605
3,thr__L_c,L-Threonine,119.12063,-0.381786,NaN,0.000000,-0.045479,NaN,-0.039411,-0.330852
4,adocbl_c,Adenosylcobalamin,1579.60636,NaN,-0.000223,-0.000352,NaN,NaN,0.000000,-0.000223
...,...,...,...,...,...,...,...,...,...,...
124,tmndnc_c,"Timnodonic acid C20:5, n-3",302.50000,-0.000044,NaN,0.000000,-0.000013,NaN,-0.000011,-0.000038
125,doco13ac_c,13Z)-13-docosenoic acid,338.60000,-0.000077,NaN,0.000000,-0.000026,NaN,-0.000023,-0.000067
126,ttc_c,Tetracosanoate n C240 C24H47O2,368.60000,-0.000035,NaN,0.000000,-0.000013,NaN,-0.000011,-0.000030
127,lnlc_c,Linoleic acid (all cis C18:2) n-6,280.40000,-0.000222,NaN,0.000000,-0.000062,NaN,-0.000054,-0.000193


In [28]:
copy_of_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0003313,0,0.00%
cl_e,EX_cl_e,0.0003313,0,0.00%
co2_e,EX_co2_e,2.5,1,100.00%
cobalt2_e,EX_cobalt2_e,6.364E-06,0,0.00%
cu2_e,EX_cu2_e,4.512E-05,0,0.00%
fe2_e,EX_fe2_e,0.0009527,0,0.00%
h2_e,EX_h2_e,15.27,0,0.00%
k_e,EX_k_e,0.01242,0,0.00%
mg2_e,EX_mg2_e,0.0005521,0,0.00%
mn2_e,EX_mn2_e,4.398E-05,0,0.00%


Change the coeff from the excel file of the already exisiting metabolite of the growth equation

In [109]:
i = 0
reaction = copy_of_model.reactions.get_by_id('Growth')
for element in pd_biomass.Biomass_equation:
    found = False
    for metabolite, coefficient in reaction.metabolites.items():
        if element == metabolite.id:
            found = True
            # Get the value of the 'normalized_mmol/g' column for the current metabolite
            normalized_mmol_g = pd_biomass.loc[pd_biomass['Biomass_equation'] == element, 'normalized_mmol/g'].values[0]
            print(f'The metabolite from the solar food BE = {element} coefficient {normalized_mmol_g}')
            print(f'The metabolite from the model = {metabolite.id} coefficient {coefficient}')
            initial_coefficient = coefficient
            # Update the coefficient of the metabolite in the reaction
            reaction.add_metabolites({metabolite: normalized_mmol_g}, combine=False)
            print(f'The coefficient was updated {initial_coefficient} ------> {coefficient}')
            i += 1
            solution = copy_of_model.slim_optimize()
            print(solution)
    if not found:
        pass
print(i)

The metabolite from the solar food BE = 10fthf_c coefficient -0.000223
The metabolite from the model = 10fthf_c coefficient -0.000223
The coefficient was updated -0.000223 ------> -0.000223
0.24372205498150215
The metabolite from the solar food BE = pydx5p_c coefficient -0.000223
The metabolite from the model = pydx5p_c coefficient -0.000223
The coefficient was updated -0.000223 ------> -0.000223
0.24372205498151733
The metabolite from the solar food BE = thr__L_c coefficient -0.33085175798343425
The metabolite from the model = thr__L_c coefficient -0.253687
The coefficient was updated -0.253687 ------> -0.253687
0.24190229624969578
The metabolite from the solar food BE = h_c coefficient 29.1804227450256
The metabolite from the model = h_c coefficient 53.95
The coefficient was updated 53.95 ------> 53.95
0.24190229624968937
The metabolite from the solar food BE = thmpp_c coefficient -0.000223
The metabolite from the model = thmpp_c coefficient -0.000223
The coefficient was updated -0.0

Compare the coeff of the excel to the one in the model

In [110]:
i=0
reaction = copy_of_model.reactions.get_by_id('Growth')
for element in pd_biomass.Biomass_equation:
    for metabolite, coefficient in reaction.metabolites.items():
        if element == metabolite.id:
            print(element)
            normalized_mmol_g = pd_biomass.loc[pd_biomass['Biomass_equation'] == element, 'normalized_mmol/g'].values[0]
            print(normalized_mmol_g)
            print(coefficient)
            i+=1
print(i)

10fthf_c
-0.000223
-0.000223
pydx5p_c
-0.000223
-0.000223
thr__L_c
-0.33085175798343425
-0.33085175798343425
h_c
29.1804227450256
29.1804227450256
thmpp_c
-0.000223
-0.000223
mn2_c
-0.000658
-0.000658
sheme_c
-0.000223
-0.000223
ser__L_c
-0.29855906692147155
-0.29855906692147155
ctp_c
-0.010737250405651103
-0.010737250405651103
arg__L_c
-0.2557979323259868
-0.2557979323259868
asp__L_c
-0.2529593526847482
-0.2529593526847482
nad_c
-0.001787
-0.001787
mg2_c
-0.008253
-0.008253
adp_c
29.1804227450256
29.1804227450256
met__L_c
-0.14267468516500864
-0.14267468516500864
pi_c
29.1804227450256
29.1804227450256
zn2_c
-0.000324
-0.000324
tyr__L_c
-0.1593258444308896
-0.1593258444308896
cys__L_c
-0.037472225353906465
-0.037472225353906465
amet_c
-0.000223
-0.000223
trp__L_c
-0.08097676930208217
-0.08097676930208217
lys__L_c
-0.25760196352578224
-0.25760196352578224
atp_c
-29.190681913288
-29.190681913288
thf_c
-0.000223
-0.000223
dttp_c
-0.013367475433835764
-0.013367475433835764
dctp_c
-0.015499

Create a dict to see which metabolite are not in the biomass equation

In [94]:
metabolite_not_found = {}
i = 0
reaction = copy_of_model.reactions.get_by_id('Growth')
for element in pd_biomass.Biomass_equation:
    found = False
    for metabolite, coefficient in reaction.metabolites.items():
        if element == metabolite.id:
            found = True
    if not found:
        normalized_mmol_g = pd_biomass.loc[pd_biomass['Biomass_equation'] == element, 'normalized_mmol/g'].values[0]
        print(f'{element:<15}  {normalized_mmol_g:>25}')
        metabolite_not_found[element] = normalized_mmol_g
        i+=1
print(i)

adocbl_c                         -0.000223
gthrd_c                          -0.000223
ni2_c                            -0.000307
q8h2_c                           -0.000223
spmd_c                           -0.006744
mococdp_c                           -7e-06
2fe2s_c                           -2.5e-05
btn_c              -1.1414549099588767e-07
4fe4s_c                          -0.000248
enter_c                          -0.000223
rmn_c                 -0.12458295393473183
arab__L_c             -0.03001579312991268
gal__L_c             -0.005772139750897219
fru_c                  -0.6445556055168562
xyl__D_c             -0.005772267909598585
man_c                -0.004810116459081017
galam_c              -0.004836694654618716
gam_c                -0.004836694654618716
retinol_c           -9.074246924272647e-09
avite1_c           -3.0180772507593555e-06
vitd3_c             -5.633064100806269e-10
vitd2_c             -5.462623432098053e-10
thm_c              -1.3154306833538052e-05
pydxn_c    

Compare metabolite from another model biomass equation to the model

In [30]:
df_met_notfound = pd.DataFrame(columns=['metabolite_id','metabolite_name'])

In [95]:
# Initialize a counter for metabolites not found and a dictionary to store them
metabolomic_data = []
i = 0

# Retrieve the 'Growth' reaction from both models
reaction = copy_of_model.reactions.get_by_id('Growth')
reaction1 = model1.reactions.get_by_id('Growth')

# Iterate over metabolites in the 'Growth' reaction of model1
for metabolite1 in reaction1.metabolites:
    found = False  # Initialize the found flag for each metabolite in reaction1
    for metabolite in reaction.metabolites:
        if metabolite1.id == metabolite.id:
            found = True
            break  # If metabolite is found, break the inner loop
    
    # If the metabolite was not found, add it to the dictionary and increment the counter
    if not found:
        metabolomic_data.append({'metabolite_id': metabolite1.id, 'metabolite_name': metabolite1.name})
        i += 1

# Print the number of metabolites not found
print(f"Number of metabolites not found: {i}")

# Create the df_metabolomic DataFrame from the list
df_met_notfound = pd.concat([df_met_notfound, pd.DataFrame(metabolomic_data)], ignore_index=True)

df_met_notfound

Number of metabolites not found: 10


,metabolite_id,metabolite_name
0,murein3px4p_p,"Two disacharide linked murein units, tripeptid..."
1,gthrd_c,Reduced glutathione
2,ni2_c,Nickel
3,accoa_c,Acetyl-CoA
4,murein4p3p_p,Two linked disacharide tetrapeptide and tripep...
5,5mthf_c,5-Methyltetrahydrofolate
6,nh4_c,Ammonium
7,chor_c,Chorismate
8,spmd_c,Spermidine
9,hemeO_c,Heme O C49H56FeN4O5


In [37]:
df_met_notfound.to_excel('C:\\Users\\adam-1y9pxi6q95ggpmx\\Desktop\\metabolitenotfound.xlsx',index=False)

In [61]:
new_metabolite = Metabolite(
    id='vitd2_c',     # Metabolite ID
    name='Vitamin D2; ergocalciferol',     # Metabolite Name
    formula='C28H44O',         # Chemical Formula
    compartment='c'            # Compartment (e.g., cytoplasm)
)

new_metabolite.charge = 0

copy_of_model.add_metabolites([new_metabolite])

print(new_metabolite.formula)

C28H44O


In [68]:
reaction = copy_of_model.reactions.get_by_id('Growth')
metabolite_to_add = 'hexedecacid_c'
reaction.add_metabolites({metabolite_to_add: -0.007033})
copy_of_model.slim_optimize()

0.056786881745949085

In [16]:
reaction = used_model.reactions.get_by_id(reaction_id_to_check)
metabolite_to_add = 'atp_c'
reaction.subtract_metabolites({metabolite_to_add: -83.30525375})
copy_of_model.slim_optimize()

-1.4893506896496426e-17

Create a dict with the metabolite that are only in the model biomass equation

In [ ]:
model_BE_metabolite = {}

# Get the 'Growth' reaction
reaction = model1.reactions.get_by_id('BIOMASS_KT2440_WT3')

# Iterate over metabolites in the 'Growth' reaction
for metabolite, coefficient in reaction.metabolites.items():
    # Check if the metabolite ID is not in the 'Biomass_equation' column
    if metabolite.id not in pd_biomass['Biomass_equation'].values:
        # Add the metabolite ID and its coefficient to the dictionary
        model_BE_metabolite[metabolite.id] = coefficient

model_BE_metabolite

In [33]:
reaction_id_to_check = 'Growth'
reaction = copy_of_model.reactions.get_by_id(reaction_id_to_check)
print(f"Reaction ID: {reaction.id}")
print(f"Name: {reaction.name}")
print(f"Equation: {reaction.reaction}")
print(f"Lower Bound: {reaction.lower_bound}")
print(f"Upper Bound: {reaction.upper_bound}")
sum = 0
#reaction.name = '(S)-2-Acetolactate pyruvate-lyase (carboxylating)'
print("\nMetabolites and Stoichiometry:")
for metabolite, coefficient in reaction.metabolites.items():
    print(f"{metabolite.id:<15}: {coefficient:>25}  Name: {metabolite.name:<60}  Charge: {metabolite.charge:>3}  Formula: {metabolite.formula}")
    sum += coefficient * metabolite.charge
print(f'\nSum charge: {sum}')
print("\nAssociated Genes:")
for gene in reaction.genes:
    print(gene.id)

Reaction ID: Growth
Name: Biomass reaction
Equation: 0.000223 10fthf_c + 0.513689 ala__L_c + 0.000223 amet_c + 0.295792 arg__L_c + 0.241055 asn__L_c + 0.241055 asp__L_c + 29.18042275 atp_c + 0.005205 ca2_c + 0.005205 cl_c + 0.000576 coa_c + 0.0001 cobalt2_c + 0.133508 ctp_c + 0.000709 cu2_c + 0.09158 cys__L_c + 0.026166 datp_c + 0.027017 dctp_c + 0.027017 dgtp_c + 0.026166 dttp_c + 0.000223 fad_c + 0.006715 fe2_c + 0.007808 fe3_c + 0.26316 gln__L_c + 0.26316 glu__L_c + 0.612638 gly_c + 0.215096 gtp_c + 29.18042275 h2o_c + 0.094738 his__L_c + 0.290529 ile__L_c + 0.195193 k_c + 0.019456 kdo2lipid4_p + 0.450531 leu__L_c + 0.343161 lys__L_c + 0.153686 met__L_c + 0.008675 mg2_c + 0.000223 mlthf_c + 0.000691 mn2_c + 0.0001 mql8_c + 0.013894 murein5px4p_p + 0.001831 nad_c + 0.000447 nadp_c + 0.017868 pe160_c + 0.045946 pe160_p + 0.054154 pe161_c + 0.02106 pe161_p + 0.185265 phe__L_c + 0.000223 pheme_c + 0.221055 pro__L_c + 0.000223 pydx5p_c + 0.000223 ribflv_c + 0.215792 ser__L_c + 0.000223 s

add metabolite to the biomass composition

In [90]:
copy_of_model = model.copy()

In [87]:
metabolite_not_found.pop('lnlc_c', None)

-0.005902952960848603

In [93]:
# Convert the dictionary to a DataFrame
df_metabolite_not_found = pd.DataFrame(list(metabolite_not_found.items()), columns=['Metabolite', 'Coefficient'])

# Save the DataFrame to an Excel file
output_file = 'C:\\Users\\adam-1y9pxi6q95ggpmx\\Desktop\\metabolite_not_found.xlsx'
df_metabolite_not_found.to_excel(output_file, index=False)

print(f"The data has been saved to {output_file}")

The data has been saved to C:\Users\adam-1y9pxi6q95ggpmx\Desktop\metabolite_not_found.xlsx


In [91]:
# Get the reaction 'Growth'
reaction = copy_of_model.reactions.get_by_id('Growth')

for metabolite_id, coefficient in metabolite_not_found.items():
    try:
        print(metabolite_id)
        # Add the metabolite to the reaction with its coefficient
        reaction.add_metabolites({metabolite_id: coefficient})
        # Optimize the model and print the result
        print(copy_of_model.slim_optimize())
    except KeyError:
        # If the metabolite is not found in the model, pass
        pass

murein3px4p_p
0.06357280701152104
accoa_c
0.06356243497722236
murein4p3p_p
0.06340848185333318
5mthf_c
0.06340130985077432
nh4_c
0.06340130985077487
chor_c
0.06339772445786857
hemeO_c
0.06338016189270824
murein4p4p_p
0.06274673112747187
mococdp_c
2dmmql8_c
0.06272917634107245
malcoa_c
0.06272800532194366
murein4px4p_p
0.062099830246387024
2fe2s_c
ptrc_c
0.06189522332031643
nadph_c
0.061884444711125974
mobd_c
0.061884444711124656
4fe4s_c
nadh_c
0.06188299712424866
murein4px4px4p_p
0.0617687781162228
succoa_c
0.061765039268234574
gal__L_c
glc__D_c
0.06070882696360335
galam_c
gam_c
retinol_c
avite1_c
vitd3_c
vitd2_c
thm_c
rbflvrd_c
0.06070746642455039
nac_c
0.06070302380473093
pnto__R_c
0.060701881750210775
phllqne_c
caro_c
clpn161_p
0.060456488053340945
pg160_p
0.059873331953638254
clpn160_p
0.05956674199177274
clpn181_p
0.05943139556552227
colipa_e
pe181_p
0.058423264500054826
pg161_p
0.05799865926385473
ttdca_c
0.05798746728729021
hexadecacid_c
hexedecacid_c
hpdca_c
octadecacid_c
octed

GAM update

In [34]:
GAM_metabolites_1 = {'atp_c': -29.18042275, 'h2o_c': -29.18042275}
GAM_metabolites_2 = {'adp_c': 29.18042275, 'pi_c': 29.18042275, 'h_c': 29.18042275}

# Get the biomass reaction
reaction = copy_of_model.reactions.get_by_id('Growth')

# Update the coefficients for GAM_metabolites_1
for element, new_coefficient in GAM_metabolites_1.items():
    metabolite = copy_of_model.metabolites.get_by_id(element)
    if metabolite in reaction.metabolites:
        print(f'Old coefficient for {element}: {reaction.metabolites[metabolite]}')
        reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
        print(f'Updated {element} coefficient to {new_coefficient}')

# Update the coefficients for GAM_metabolites_2
for element, new_coefficient in GAM_metabolites_2.items():
    metabolite = copy_of_model.metabolites.get_by_id(element)
    if metabolite in reaction.metabolites:
        print(f'Old coefficient for {element}: {reaction.metabolites[metabolite]}')
        reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
        print(f'Updated {element} coefficient to {new_coefficient}')

# Optimize the model
solution = copy_of_model.slim_optimize()
print(f'Optimal solution: {solution}')

Old coefficient for atp_c: -54.124831
Updated atp_c coefficient to -29.18042275
Old coefficient for h2o_c: -48.601527
Updated h2o_c coefficient to -29.18042275
Old coefficient for adp_c: 53.95
Updated adp_c coefficient to 29.18042275
Old coefficient for pi_c: 53.945662
Updated pi_c coefficient to 29.18042275
Old coefficient for h_c: 53.95
Updated h_c coefficient to 29.18042275
Optimal solution: 0.0636423240292905


# Update new model

### Update charges from excel

In [11]:
file_path = 'C:\\Users\\benammara\\Documents\\PhD\\Data\\Models\\Manual Curation\\Modified reactions.xlsx'
file_path_1 = 'C:\\Users\\benammara\\Documents\\PhD\\Data\\Models\\Manual Curation\\Modified formula.xlsx'

excel_data_charge = pd.read_excel(file_path)

# Iterate over each component in the Excel file
for component in excel_data_charge['metabolite changed']:
    # Check if a metabolite is present in the model's metabolites
    metabolite_id = component
    if metabolite_id in copy_of_model.metabolites:
        # Get the metabolite from the model
        metabolite = copy_of_model.metabolites.get_by_id(metabolite_id)
        
        # Change the charge of the metabolite to a new value from the Excel file
        new_charge = int(excel_data_charge.loc[excel_data_charge['metabolite changed'] == metabolite_id, 'New charge'].values[0])
        metabolite.charge = new_charge
        
        # Print information about the modified metabolite
        print(f"Modified metabolite: {metabolite.id}")
        print(f"Updated charge: {metabolite.charge}")
    else:
        print("Metabolite not found in the model.")

Modified metabolite: 2agpg161_c
Updated charge: -1
Modified metabolite: r3hbACP_c
Updated charge: -1
Modified metabolite: tmrs2eACP_c
Updated charge: -1
Modified metabolite: 4hoxpac_p
Updated charge: -1
Modified metabolite: 4oxptn_p
Updated charge: -1
Modified metabolite: 4oxptn_e
Updated charge: -1
Modified metabolite: 4hoxpac_e
Updated charge: -1
Modified metabolite: 6atha_e
Updated charge: -1
Metabolite not found in the model.
Modified metabolite: hethmpp_c
Updated charge: -2
Modified metabolite: Pald_c
Updated charge: -2
Modified metabolite: S2hglut_c
Updated charge: -2
Modified metabolite: sdhlam_c
Updated charge: -1
Modified metabolite: pqqh2_p
Updated charge: -3
Modified metabolite: ppad_c
Updated charge: -1
Modified metabolite: cmcbtt_c
Updated charge: 1
Modified metabolite: fcmcbtt_c
Updated charge: 4
Modified metabolite: R_3hpt_p
Updated charge: -1
Modified metabolite: R_3hpt_e
Updated charge: -1
Modified metabolite: R_3hpba_p
Updated charge: -1
Modified metabolite: R_3hpba_e

In [23]:
print(len(cobra.manipulation.validate.check_mass_balance(copy_of_model)))
list_reactions = cobra.manipulation.validate.check_mass_balance(copy_of_model)
for reaction, balance_dict in list_reactions.items():
    reaction_id = reaction.id
    charge_balance = balance_dict
    print(f"Reaction ID: {reaction_id}, Charge Balance: {charge_balance}")

5
Reaction ID: DHNAOT, Charge Balance: {'charge': 2.0}
Reaction ID: HDECH, Charge Balance: {'charge': 2.0}
Reaction ID: MECDPDH4E, Charge Balance: {'charge': -1.0}
Reaction ID: P9DH, Charge Balance: {'charge': 4.0}
Reaction ID: THZSN_1, Charge Balance: {'H': -1.0}


### Create a list of reaction to update with the related model

In [15]:
import requests
list_reactions = cobra.manipulation.validate.check_mass_balance(copy_of_model)

# Dictionary to store reaction ID and associated model
reaction_model_mapping = {}

for reaction, balance_dict in list_reactions.items():
    reaction_id = reaction.id  # Get the ID of the reaction
    # URL of the API endpoint for the reaction
    url = f'http://bigg.ucsd.edu/api/v2/universal/reactions/{reaction_id}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        # Extract the model ID from the JSON data
        model_of_reaction = data['models_containing_reaction'][0]['bigg_id']
        # Store the mapping of reaction ID to its model in the dictionary
        reaction_model_mapping[reaction_id] = model_of_reaction
    else:
        # If the request was not successful, print the status code
        print("Failed to retrieve data. Status code:", response.status_code)
print(reaction_model_mapping)

{'1P2CBXLCYCL': 'iJN1463', '1P2CBXLR': 'iJN1463', '3HAACOAT140': 'iJN1463', '3HACPH': 'iYS854', '3HAD40': 'iAF1260', '3HOXTPP': 'iYS854', '3OAR40': 'iAF1260', '4HDPROO': 'iJN1463', 'AACLAM': 'iYS854', 'AACPS3': 'iAF1260', 'AACPS5': 'iAF1260', 'AADa': 'iCN718', 'AADb': 'iCN718', 'AALDH': 'iNF517', 'ACGAMPM': 'iAM_Pb448', 'ACHBS': 'iAF1260', 'ACHBS2': 'iSynCJ816', 'ACM6PH': 'iAF1260', 'ACMANApts': 'iJR904', 'ACOAD10f': 'iJN1463', 'ACOAD11f': 'iJN1463', 'ACOAD12f': 'iJN1463', 'ACOAD13f': 'iJN1463', 'ACOAD14f': 'iJN1463', 'ACOAD15f': 'iJN1463', 'ACOAD16f': 'iJN1463', 'ACOAD17f': 'iJN1463', 'ACOAD18f': 'iJN1463', 'ACOAD19f': 'iJN1463', 'ACOAD20f': 'iJN1463', 'ACOAD21f': 'iJN1463', 'ACOAD22f': 'iJN1463', 'ACOAD23f': 'iJN1463', 'ACOAD29f': 'iJN1463', 'ACOAD3': 'iAPECO1_1312', 'ACOAD30f': 'iJN1463', 'ACOAD31f': 'iJN1463', 'ACOAD33f': 'iJN1463', 'ACOAD3f': 'iAF1260', 'ACOAD6': 'iAPECO1_1312', 'ACOAD6f': 'iAF1260', 'ACOAM': 'iYS854', 'ACPPAT140': 'iAF987', 'ACPPAT160': 'iAF987', 'ACPPAT181': 'iA

### Update charge from BIGG API

In [18]:
for reaction, model_name in reaction_model_mapping.items():
    reaction_to_modify = copy_of_model.reactions.get_by_id(reaction)
    for metabolite, coefficient in reaction_to_modify.metabolites.items():
        if metabolite.charge == 0:
            url = f'http://bigg.ucsd.edu/api/v2/models/{model_name}/metabolites/{metabolite.id}'
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                # Check if the charge information is available and matches the current metabolite charge
                if data['charge'] is not None and data['charge'] != metabolite.charge:
                    print(metabolite.id)
                    print(f' metabolite charge : {metabolite.charge}')
                    print(f' bigg metabolite charge : {data['charge']}')
                    metabolite.charge = data['charge']
            else:
                print("Failed to retrieve data. Status code:", response.status_code)

1p2cbxl_c
 metabolite charge : 0
 bigg metabolite charge : -1
3hmrsACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
but2eACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
actACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
palmACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
octeACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
acgam1p_c
 metabolite charge : 0
 bigg metabolite charge : -2
2ahbut_c
 metabolite charge : 0
 bigg metabolite charge : -1
acmum6p_c
 metabolite charge : 0
 bigg metabolite charge : -3
acmanap_c
 metabolite charge : 0
 bigg metabolite charge : -2
fad_c
 metabolite charge : 0
 bigg metabolite charge : -2
fadh2_c
 metabolite charge : 0
 bigg metabolite charge : -2
myrsACP_c
 metabolite charge : 0
 bigg metabolite charge : -1
appl_c
 metabolite charge : 0
 bigg metabolite charge : 1
r5p_c
 metabolite charge : 0
 bigg metabolite charge : -2
adsel_c
 metabolite charge : 0
 bigg metabolite charge : -2
3padsel_c
 metabolite ch

### Update charge and formula from a curated model

In [65]:
model1 = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\Models\\Manual Curation\\updated_growth_gapfilled_autotrophic_1.sbml')

In [66]:
list_reactions = cobra.manipulation.validate.check_mass_balance(copy_of_model)
copy_of_model1 = model1.copy()
# Iterate through metabolites in the non-curated model
for metabolite in copy_of_model.metabolites:
    # Check if the metabolite exists in the curated model
    if metabolite.id in copy_of_model1.metabolites:
        # Find the corresponding metabolite in the curated model
        curated_metabolite = copy_of_model1.metabolites.get_by_id(metabolite.id)
        # Update the charge and formula of the non-curated metabolite
        metabolite.charge = curated_metabolite.charge
        metabolite.formula = curated_metabolite.formula
    else:
        print(f"Metabolite '{metabolite.id}' not found in the curated model.")


Metabolite 'etoh_e' not found in the curated model.


### Update metabolite with coa substring

In [16]:
substring_to_check = 'coa'
for metabolite in copy_of_model.metabolites:
    if substring_to_check in metabolite.id:
        if metabolite.charge == 0:
            metabolite.charge = -4
            # Print information about the modified metabolite
            print(f"Modified metabolite: {metabolite.id}")
            print(f"Updated charge of : {metabolite.charge}")

Modified metabolite: 2maacoa_c
Updated charge of : -4
Modified metabolite: 3hhd710coa_c
Updated charge of : -4
Modified metabolite: 3hhdccoa_c
Updated charge of : -4
Modified metabolite: 3hhdd4coa_c
Updated charge of : -4
Modified metabolite: 3hlnlccoa_c
Updated charge of : -4
Modified metabolite: 3hocoa_c
Updated charge of : -4
Modified metabolite: 3hodcoa_c
Updated charge of : -4
Modified metabolite: 3mgcoa_c
Updated charge of : -4
Modified metabolite: 3ohd710coa_c
Updated charge of : -4
Modified metabolite: 3ohdccoa_c
Updated charge of : -4
Modified metabolite: 3ohdd4coa_c
Updated charge of : -4
Modified metabolite: 3olnlccoa_c
Updated charge of : -4
Modified metabolite: benzcoa_c
Updated charge of : -4
Modified metabolite: ctncoa_c
Updated charge of : -4
Modified metabolite: hd2coa_c
Updated charge of : -4
Modified metabolite: hd710_2_coa_c
Updated charge of : -4
Modified metabolite: hd_7_10_coa_c
Updated charge of : -4
Modified metabolite: hdd4_2_coa_c
Updated charge of : -4
Modif

I noticed that carveme doesn't provide the appropriate charge for metabolites with Coa molecules

# Create a list of all reactions

In [ ]:
df_reactions = pd.DataFrame(columns=['reaction_id','reaction_name','reaction_equation','reaction_lower_bound','reaction_upper_bound','reaction_gene','reaction_annotation'])

In [16]:
# Iterate over all reactions in the model and populate the DataFrame
for reaction in copy_of_model.reactions:
    # Create a DataFrame for the current reaction
    reaction_data = pd.DataFrame([{
        "reaction_id": reaction.id,
        "reaction_name": reaction.name,
        "reaction_equation": reaction.reaction,
        "reaction_lower_bound": reaction.lower_bound,
        "reaction_upper_bound": reaction.upper_bound,
        "reaction_gene": reaction.gene_reaction_rule,
        "reaction_annotation": reaction.annotation
    }])
    
    # Concatenate the current reaction's DataFrame to the main DataFrame
    df_reactions = pd.concat([df_reactions, reaction_data], ignore_index=True)

# Print the resulting DataFrame to verify the results
df_reactions.to_excel('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\reaction_list.xlsx',index=False)


# Create a list of all metabolites

In [18]:
df_metabolites = pd.DataFrame(columns=['metabolite_id','metabolite_name','metabolite_formula','metabolite_charge','related_reactions'])

In [19]:
# Iterate over all reactions in the model and populate the DataFrame
for metabolite in copy_of_model.metabolites:
    # Create a DataFrame for the current reaction
    metabolite_data = pd.DataFrame([{
        "metabolite_id": metabolite.id,
        "metabolite_name": metabolite.name,
        "metabolite_formula": metabolite.formula,
        "metabolite_charge": metabolite.charge,
        "related_reactions": list(metabolite.reactions),
    }])
    # Concatenate the current reaction's DataFrame to the main DataFrame
    df_metabolites = pd.concat([df_metabolites, metabolite_data], ignore_index=True)

# Print the resulting DataFrame to verify the results
df_metabolites.to_excel('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\metabolites_list.xlsx',index=False)

# Create a list of all metabolites in a reaction

In [97]:
df_reaction2 = pd.DataFrame(columns=['met_id','met_name','met_coeff','met_charge'])

In [98]:
reaction_id_to_check = 'Growth'
reaction = copy_of_model.reactions.get_by_id(reaction_id_to_check)
for metabolite, coefficient in reaction.metabolites.items():
    print(f"{metabolite.id:<15}: {coefficient:>25}  Name: {metabolite.name:<60}  Charge: {metabolite.charge:>3}  Formula: {metabolite.formula}")
    # Create a DataFrame for the current reaction
    reaction_data = pd.DataFrame([{
        "met_id": metabolite.id,
        "met_name": metabolite.name,
        "met_coeff": coefficient,
        "met_charge": metabolite.charge,
    }])
    # Concatenate the current reaction's DataFrame to the main DataFrame
    df_reaction2 = pd.concat([df_reaction2, reaction_data], ignore_index=True)

df_reaction2.to_excel('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\Data\\growth_met_list_updated.xlsx',index=False)

10fthf_c       :                 -0.000223  Name: 10-Formyltetrahydrofolate                                     Charge:  -2  Formula: C20H21N7O7
ala__L_c       :                 -0.513689  Name: L-Alanine                                                     Charge:   0  Formula: C3H7NO2
amet_c         :                 -0.000223  Name: S-Adenosyl-L-methionine                                       Charge:   1  Formula: C15H23N6O5S
arg__L_c       :                 -0.295792  Name: L-Arginine                                                    Charge:   1  Formula: C6H15N4O2
asn__L_c       :                 -0.241055  Name: L-Asparagine                                                  Charge:   0  Formula: C4H8N2O3
asp__L_c       :                 -0.241055  Name: L-Aspartate                                                   Charge:  -1  Formula: C4H6NO4
atp_c          :              -29.18042275  Name: ATP C10H12N5O13P3                                             Charge:  -4  Formula: C10H

C:\Users\adam-1y9pxi6q95ggpmx\AppData\Local\Temp\ipykernel_15784\98607269.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_reaction2 = pd.concat([df_reaction2, reaction_data], ignore_index=True)
